In [ ]:
import os
import gc
import zipfile
import getpass
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from arabert.preprocess import ArabertPreprocessor

import scipy.optimize as optimize
from functools import partial

# ==========================================================
# 1. ĐỊNH NGHĨA CORAL TRAINER & METRICS
# ==========================================================
class CORALTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        device = logits.device
        
        # Ma trận Target cho 18 lằn ranh
        target = torch.zeros_like(logits, device=device)
        for k in range(logits.shape[1]):
            target[:, k] = (labels > k).float()
            
        loss_per_sample = F.binary_cross_entropy_with_logits(
            logits, target, reduction='none'
        ).mean(dim=1)
        
        # Ép trọng số mẫu
        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels]
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_coral(eval_pred):
    logits, labels = eval_pred
    preds_binary = logits > 0
    pred_labels = preds_binary.sum(axis=1)
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    return {"qwk": qwk}

# ==========================================================
# 2. TẢI TẬP BLIND TEST BẢO MẬT & TIỀN XỬ LÝ
# ==========================================================
print("🔒 Vui lòng nhập Hugging Face Access Token:")
access_token = getpass.getpass("Token: ")

print("📥 Đang tải tập Blind Test từ Hugging Face...")
barec_sent = load_dataset("CAMeL-Lab/BAREC-Shared-Task-2026-BlindTest-sent", token=access_token)
df_blind_test = barec_sent['train'].to_pandas()

print("🧹 Đang tiền xử lý (AraBERT Preprocessor) tập Blind Test...")
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)
df_blind_test['Sentence_Normalized'] = df_blind_test['Sentence'].apply(lambda x: arabert_prep.preprocess(str(x)))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    # LƯU Ý: Đang để max_length = 128. Nếu VRAM của bạn gánh được, hãy nâng lên 256 để model đọc đủ các câu dài.
    return tokenizer(examples["Sentence_Normalized"], padding="max_length", truncation=True, max_length=128)

hf_blind_test = Dataset.from_pandas(df_blind_test[['Sentence_Normalized']])
tokenized_blind_test = hf_blind_test.map(tokenize_function, batched=True)
tokenized_blind_test.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# ==========================================================
# 3. GỘP DỮ LIỆU & CHUẨN BỊ K-FOLD
# ==========================================================
print("\n📦 ĐANG GỘP TẬP TRAIN + VALID + OPEN TEST CŨ ĐỂ TỐI ĐA HÓA DỮ LIỆU...")
PROCESSED_DIR = '../data/processed/'
df_train_old = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid_old = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')
df_test_old = pd.read_parquet(PROCESSED_DIR + 'test_cleaned.parquet')

# Ép kiểu và gộp toàn bộ
df_train_old['label'] = df_train_old['label'].astype(np.int64)
df_valid_old['label'] = df_valid_old['label'].astype(np.int64)
df_test_old['label'] = df_test_old['label'].astype(np.int64) # Test cũ cũng có nhãn, lấy dùng luôn!
df_all = pd.concat([df_train_old, df_valid_old, df_test_old], ignore_index=True)

# Khởi tạo 5-Fold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# ==========================================================
# 4. VÒNG LẶP HUẤN LUYỆN 5-FOLD & LƯU OOF LOGITS
# ==========================================================
test_logits_folds = []
oof_logits = np.zeros((len(df_all), 18)) # Hứng logits OOF
oof_true_labels = np.zeros(len(df_all))  # Hứng nhãn OOF

for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['label'])):
    print(f"\n========================================")
    print(f"🚀 BẮT ĐẦU HUẤN LUYỆN FOLD {fold + 1}/{n_splits}")
    print(f"========================================")
    
    train_fold = df_all.iloc[train_idx]
    val_fold = df_all.iloc[val_idx]
    
    # Class Weights chuyên biệt cho Fold
    cw = compute_class_weight('balanced', classes=np.arange(19), y=train_fold['label'].values)
    cw_clipped = np.clip(cw, 0.5, 5.0)
    cw_normalized = cw_clipped / cw_clipped.mean()
    class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)
    
    # Dataset & Format
    hf_train = Dataset.from_pandas(train_fold[['Sentence_Normalized', 'label']])
    hf_val = Dataset.from_pandas(val_fold[['Sentence_Normalized', 'label']])
    
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    tokenized_val = hf_val.map(tokenize_function, batched=True)
    tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    tokenized_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    
    # Khởi tạo LẠI Mô hình & LoRA
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=18)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["query", "value", "dense"] 
    )
    model = get_peft_model(model, lora_config)
    
    output_dir = f"../saved_models/arabert_lora_coral_fold_{fold+1}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=3e-4, per_device_train_batch_size=8,
        gradient_accumulation_steps=4, per_device_eval_batch_size=16,
        num_train_epochs=5, weight_decay=0.01,
        load_best_model_at_end=True, metric_for_best_model="qwk", greater_is_better=True,
        bf16=True, fp16=False, seed=42
    )
    
    trainer = CORALTrainer(
        model=model, args=training_args,
        train_dataset=tokenized_train, eval_dataset=tokenized_val,
        compute_metrics=compute_metrics_coral,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights_tensor
    )
    
    # Train
    trainer.train()
    
    # Lấy Logits OOF (Tập Validation)
    print(f"🔍 Đang trích xuất Logits Validation (OOF) từ Fold {fold + 1}...")
    val_pred = trainer.predict(tokenized_val)
    oof_logits[val_idx] = val_pred.predictions[:, :18]
    oof_true_labels[val_idx] = val_pred.label_ids.astype(int)
    
    # Lấy Logits Blind Test
    print(f"🔍 Đang trích xuất Logits Blind Test từ Fold {fold + 1}...")
    test_pred = trainer.predict(tokenized_blind_test)
    test_logits = test_pred.predictions[:, :18]
    test_logits_folds.append(test_logits)
    
    # DỌN RÁC VRAM
    del model, trainer, training_args
    gc.collect()
    torch.cuda.empty_cache()
    print(f"🧹 Đã dọn dẹp VRAM, sẵn sàng cho Fold tiếp theo.\n")

# ==========================================================
# 5. DÒ TÌM DELTA TỐI ƯU & XUẤT FILE NỘP BÀI BLIND TEST
# ==========================================================
print("\n👑 ĐANG TIẾN HÀNH DÒ TÌM DELTA TỐI ƯU TRÊN TẬP OOF (KHÔNG BỊ LEAKAGE)...")
best_delta = 0.0
best_qwk = -1.0

for delta in np.arange(-2.0, 2.0, 0.01):
    pred_labels = (oof_logits > delta).sum(axis=1)
    qwk = cohen_kappa_score(oof_true_labels, pred_labels, weights='quadratic')
    if qwk > best_qwk:
        best_qwk = qwk
        best_delta = delta

print(f"🎯 DELTA VÀNG (Từ OOF) : {best_delta:.2f}")
print(f"🚀 QWK OOF K-FOLD      : {best_qwk:.4f}")

print("\n🔮 ĐANG ÁP DỤNG DELTA VÀNG LÊN TẬP BLIND TEST VÀ ĐÓNG GÓI...")
avg_test_logits = np.mean(test_logits_folds, axis=0)

# Chú ý: Cột ID thường có tên là 'ID' hoặc 'Sentence ID', bạn kiểm tra lại df_blind_test nếu cần.
test_ids = df_blind_test['ID'].values 

final_preds = (avg_test_logits > best_delta).sum(axis=1) + 1
submission_df = pd.DataFrame({
    'Sentence ID': test_ids,
    'Prediction': final_preds
})

submission_df.to_csv('prediction', index=False, lineterminator='\n')
with zipfile.ZipFile('prediction_kfold_blind.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')

print("✅ File 'prediction_kfold_blind.zip' đã xuất xưởng thành công!")

🔒 Vui lòng nhập Hugging Face Access Token:
📥 Đang tải tập Blind Test từ Hugging Face...
🧹 Đang tiền xử lý (AraBERT Preprocessor) tập Blind Test...


Map:   0%|          | 0/8077 [00:00<?, ? examples/s]


📦 ĐANG GỘP TẬP TRAIN + VALID + OPEN TEST CŨ ĐỂ TỐI ĐA HÓA DỮ LIỆU...

🚀 BẮT ĐẦU HUẤN LUYỆN FOLD 1/5


Map:   0%|          | 0/55377 [00:00<?, ? examples/s]

Map:   0%|          | 0/13845 [00:00<?, ? examples/s]

...

{'eval_loss': 0.05906056985259056, 'eval_qwk': 0.8455552558230854, 'eval_runtime': 58.3485, 'eval_samples_per_second': 237.264, 'eval_steps_per_second': 14.842, 'epoch': 5.0}
{'train_runtime': 3825.5423, 'train_samples_per_second': 72.379, 'train_steps_per_second': 2.261, 'train_loss': 0.04989909861129143, 'epoch': 5.0}
🔍 Đang trích xuất Logits Validation (OOF) từ Fold 5...


  0%|          | 0/866 [00:00<?, ?it/s]

🔍 Đang trích xuất Logits Blind Test từ Fold 5...


  0%|          | 0/505 [00:00<?, ?it/s]

🧹 Đã dọn dẹp VRAM, sẵn sàng cho Fold tiếp theo.


👑 ĐANG TIẾN HÀNH DÒ TÌM DELTA TỐI ƯU TRÊN TẬP OOF (KHÔNG BỊ LEAKAGE)...
🎯 DELTA VÀNG (Từ OOF) : -0.41
🚀 QWK OOF K-FOLD      : 0.8450

🔮 ĐANG ÁP DỤNG DELTA VÀNG LÊN TẬP BLIND TEST VÀ ĐÓNG GÓI...
✅ File 'prediction_kfold_blind.zip' đã xuất xưởng thành công!


In [ ]:
import os
import gc
import zipfile
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# ==========================================================
# 1. BẢO LƯU KẾT QUẢ SEED 42 (TỪ CELL TRƯỚC ĐÓ)
# ==========================================================
print("💾 Đang sao lưu Logits của Seed 42...")
oof_logits_seed42 = oof_logits.copy()
test_logits_seed42 = avg_test_logits.copy()

# ==========================================================
# 2. KHỞI TẠO & CHẠY 5-FOLD CHO SEED 123
# ==========================================================
test_logits_folds_seed123 = []
oof_logits_seed123 = np.zeros((len(df_all), 18))

print("\n========================================================")
print("🌟 BẮT ĐẦU CHIẾN DỊCH SEED ENSEMBLE (SEED = 123) 🌟")
print("========================================================")

for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['label'])):
    print(f"\n🚀 BẮT ĐẦU HUẤN LUYỆN FOLD {fold + 1}/{n_splits} (SEED 123)")
    print(f"========================================")
    
    train_fold = df_all.iloc[train_idx]
    val_fold = df_all.iloc[val_idx]
    
    # Class weights
    cw = compute_class_weight('balanced', classes=np.arange(19), y=train_fold['label'].values)
    cw_clipped = np.clip(cw, 0.5, 5.0)
    cw_normalized = cw_clipped / cw_clipped.mean()
    class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)
    
    # HuggingFace Datasets
    hf_train = Dataset.from_pandas(train_fold[['Sentence_Normalized', 'label']])
    hf_val = Dataset.from_pandas(val_fold[['Sentence_Normalized', 'label']])
    
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    tokenized_val = hf_val.map(tokenize_function, batched=True)
    tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    tokenized_val.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    
    # Khởi tạo mô hình & LoRA
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=18)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["query", "value", "dense"] 
    )
    model = get_peft_model(model, lora_config)
    
    # 🎯 SỬA SEED = 123 VÀ OUTPUT DIR KHÁC
    output_dir = f"../saved_models/arabert_lora_coral_fold_{fold+1}_seed123"
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=3e-4, per_device_train_batch_size=8,
        gradient_accumulation_steps=4, per_device_eval_batch_size=16,
        num_train_epochs=5, weight_decay=0.01,
        load_best_model_at_end=True, metric_for_best_model="qwk", greater_is_better=True,
        bf16=True, fp16=False, 
        seed=123  # <-- Seed mới
    )
    
    trainer = CORALTrainer(
        model=model, args=training_args,
        train_dataset=tokenized_train, eval_dataset=tokenized_val,
        compute_metrics=compute_metrics_coral,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights_tensor
    )
    
    trainer.train()
    
    # Trích xuất OOF
    print(f"🔍 Đang trích xuất Logits Validation (OOF) Fold {fold + 1} (Seed 123)...")
    val_pred = trainer.predict(tokenized_val)
    oof_logits_seed123[val_idx] = val_pred.predictions[:, :18]
    
    # Trích xuất Blind Test
    print(f"🔍 Đang trích xuất Logits Blind Test Fold {fold + 1} (Seed 123)...")
    test_pred = trainer.predict(tokenized_blind_test)
    test_logits_folds_seed123.append(test_pred.predictions[:, :18])
    
    # Dọn VRAM
    del model, trainer, training_args
    gc.collect()
    torch.cuda.empty_cache()
    print(f"🧹 Đã dọn dẹp VRAM, sẵn sàng cho Fold tiếp theo.\n")


# ==========================================================
# 3. TRỘN LOGITS (BLEND) & DÒ TÌM 2-PARAM THRESHOLD
# ==========================================================
print("\n👑 ĐANG TRỘN LOGITS TỪ SEED 42 VÀ SEED 123...")
blended_oof_logits = (oof_logits_seed42 + oof_logits_seed123) / 2.0
avg_test_logits_seed123 = np.mean(test_logits_folds_seed123, axis=0)
blended_test_logits = (test_logits_seed42 + avg_test_logits_seed123) / 2.0

💾 Đang sao lưu Logits của Seed 42...

🌟 BẮT ĐẦU CHIẾN DỊCH SEED ENSEMBLE (SEED = 123) 🌟

🚀 BẮT ĐẦU HUẤN LUYỆN FOLD 1/5 (SEED 123)


Map:   0%|          | 0/55377 [00:00<?, ? examples/s]

Map:   0%|          | 0/13845 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/8650 [00:00<?, ?it/s]

...

{'eval_loss': 0.06052587181329727, 'eval_qwk': 0.8434845387715101, 'eval_runtime': 58.599, 'eval_samples_per_second': 236.25, 'eval_steps_per_second': 14.778, 'epoch': 5.0}
{'train_runtime': 3814.5953, 'train_samples_per_second': 72.587, 'train_steps_per_second': 2.268, 'train_loss': 0.04951145111480889, 'epoch': 5.0}
🔍 Đang trích xuất Logits Validation (OOF) Fold 5 (Seed 123)...


  0%|          | 0/866 [00:00<?, ?it/s]

🔍 Đang trích xuất Logits Blind Test Fold 5 (Seed 123)...


  0%|          | 0/505 [00:00<?, ?it/s]

🧹 Đã dọn dẹp VRAM, sẵn sàng cho Fold tiếp theo.


👑 ĐANG TRỘN LOGITS TỪ SEED 42 VÀ SEED 123...


In [7]:
import numpy as np
import pandas as pd
import zipfile
from sklearn.metrics import cohen_kappa_score
from scipy.special import expit  # Dùng để tính xác suất sigmoid

# ==========================================================
# 1. DÒ TỶ LỆ TRỘN VÀNG (SEED 42 & SEED 123) TRÊN OOF
# ==========================================================
print("⚖️ BƯỚC 1: DÒ TỶ LỆ TRỘN (SEED 42 & SEED 123) TRÊN OOF...")
best_w = 0.5
best_qwk_pre = -1.0

# Thử nghiệm trọng số từ 0% đến 100% cho Seed 42
for w in np.arange(0.0, 1.05, 0.05):
    temp_blend = w * oof_logits_seed42 + (1 - w) * oof_logits_seed123
    # Dùng global delta tạm là -0.44 (từ các bước trước) để tìm w
    preds = (temp_blend > best_delta).sum(axis=1)
    qwk = cohen_kappa_score(oof_true_labels, preds, weights='quadratic')
    if qwk > best_qwk_pre:
        best_qwk_pre = qwk
        best_w = w

print(f"🎯 Tỷ lệ Seed tối ưu: {best_w*100:.0f}% Seed42 + {(1-best_w)*100:.0f}% Seed123")

# Chốt sổ mảng Logits sau khi trộn
blended_oof_logits = best_w * oof_logits_seed42 + (1 - best_w) * oof_logits_seed123
blended_test_logits = best_w * test_logits_seed42 + (1 - best_w) * avg_test_logits_seed123

# ==========================================================
# 2. TÌM 2-PARAM SPLIT (dl, dh) TRÊN BLENDED OOF
# ==========================================================
print("\n🎯 BƯỚC 2: TÌM 2-PARAM SPLIT (dl, dh) TRÊN BLENDED OOF...")
best_qwk_blend, best_dl, best_dh = -1.0, 0.0, 0.0

def kappa_loss(coef, X, y):
    preds = (X > coef).sum(axis=1)
    return -cohen_kappa_score(y, preds, weights='quadratic')

print("👑 ĐANG TỐI ƯU 18 THRESHOLD BẰNG NELDER-MEAD...")
result = optimize.minimize(
    partial(kappa_loss, X=blended_oof_logits, y=oof_true_labels.astype(int)),
    x0=np.zeros(18),          # Khởi điểm tại 0.0 cho cả 18 chiều
    method='nelder-mead',
    options={'xatol': 1e-4, 'fatol': 1e-4, 'maxiter': 50000}
)
best_thresholds = result['x']
best_qwk_blend = -result.fun

print(f"🏆 QWK OOF Nelder-Mead : {best_qwk_blend:.4f}")
print(f"🎯 18 Thresholds       : {np.round(best_thresholds, 3)}")

# ==========================================================
# 3. ÉP PHÂN PHỐI (DISTRIBUTION ALIGNMENT) LÊN BLIND TEST
# ==========================================================
print("\n🔮 BƯỚC 3: ÁP DỤNG ÉP PHÂN PHỐI LÊN BLIND TEST VÀ XUẤT FILE...")

# Dùng Sigmoid lên Logits để lấy xác suất, rồi cộng dồn. 
# Kết quả tạo ra một thang điểm liên tục (continuous score) để xếp hạng các câu từ dễ nhất đến khó nhất.
test_probs = expit(blended_test_logits)
continuous_scores = test_probs.sum(axis=1) 

# Lấy tỷ lệ phân phối gốc thực tế từ tập OOF (Train + Valid + Test cũ)
label_counts = pd.Series(oof_true_labels).value_counts(normalize=True).sort_index()

# Tính quota: Số lượng câu tối đa cho từng Level trong tập Blind Test (8077 câu)
n_test = len(blended_test_logits)
target_counts = (label_counts * n_test).round().astype(int)

# Xử lý sai số làm tròn để đảm bảo tổng đúng bằng 8077
diff = n_test - target_counts.sum()
target_counts.iloc[-1] += diff 

# Sắp xếp index của Blind Test từ câu DỄ NHẤT đến câu KHÓ NHẤT
sorted_indices = np.argsort(continuous_scores)

# Điền nhãn theo đúng quota của phân phối gốc
final_preds = np.zeros(n_test, dtype=int)
current_idx = 0
for label in range(19): # Các nhãn từ 0 đến 18
    count = target_counts.iloc[label]
    if count > 0:
        assigned_indices = sorted_indices[current_idx : current_idx + count]
        final_preds[assigned_indices] = label + 1 # Chuyển về thang 1-19 của cuộc thi
        current_idx += count

print("✅ Đã ép phân phối thành công! Chấm dứt tình trạng model 'nhát tay'.")

# ==========================================================
# 4. ĐÓNG GÓI CHUẨN CODABENCH
# ==========================================================
# Tạo submission riêng từ Nelder-Mead để so sánh
final_preds_nm = (blended_test_logits > best_thresholds).sum(axis=1) + 1
final_preds_nm = np.clip(final_preds_nm, 1, 19)

submission_df_nm = pd.DataFrame({
    'Sentence ID': df_blind_test['ID'].values,
    'Prediction': final_preds_nm
})
submission_df_nm.to_csv('prediction', index=False, lineterminator='\n')
with zipfile.ZipFile('prediction_nelder_mead.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')
print("📦 prediction_nelder_mead.zip — submit để compare với aligned!")

⚖️ BƯỚC 1: DÒ TỶ LỆ TRỘN (SEED 42 & SEED 123) TRÊN OOF...
🎯 Tỷ lệ Seed tối ưu: 45% Seed42 + 55% Seed123

🎯 BƯỚC 2: TÌM 2-PARAM SPLIT (dl, dh) TRÊN BLENDED OOF...
👑 ĐANG TỐI ƯU 18 THRESHOLD BẰNG NELDER-MEAD...
🏆 QWK OOF Nelder-Mead : 0.8468
🎯 18 Thresholds       : [ 0.  0. -0.  0. -0. -0.  0. -0.  0. -0. -0. -0.  0.  0.  0. -0.  0.  0.]

🔮 BƯỚC 3: ÁP DỤNG ÉP PHÂN PHỐI LÊN BLIND TEST VÀ XUẤT FILE...
✅ Đã ép phân phối thành công! Chấm dứt tình trạng model 'nhát tay'.
📦 prediction_nelder_mead.zip — submit để compare với aligned!


In [24]:
import numpy as np
import pandas as pd
import zipfile
import re
import scipy.optimize as optimize
from functools import partial
from sklearn.metrics import cohen_kappa_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

print("🚀 ĐANG TÁI TẠO LẠI CÁC PHƯƠNG ÁN SUBMIT TỪ LOGITS...\n")

# ==========================================================
# CHUẨN BỊ NGUYÊN LIỆU CHUNG
# ==========================================================
# 1. 36D Features (Kết hợp Logits của Seed 42 và Seed 123)
X_oof = np.hstack([oof_logits_seed42, oof_logits_seed123])
X_test = np.hstack([test_logits_seed42, avg_test_logits_seed123])
y_oof = oof_true_labels.astype(int) # Nhãn 0-18

# 2. Scale dữ liệu cho các mô hình Meta
scaler = StandardScaler()
X_oof_scaled = scaler.fit_transform(X_oof)
X_test_scaled = scaler.transform(X_test)

test_ids = df_blind_test['ID'].values

# ==========================================================
# PHƯƠNG ÁN 1: SOFT BLEND 50/50 (CỨNG + MỀM)
# ==========================================================
# Kết hợp 50% Nelder-Mead (18-param) và 50% Ép phân phối
blended_labels_5050 = 0.5 * final_preds_nm + 0.5 * final_preds_aligned
final_preds_5050 = np.clip(np.round(blended_labels_5050).astype(int), 1, 19)

pd.DataFrame({'Sentence ID': test_ids, 'Prediction': final_preds_5050}) \
  .to_csv('prediction', index=False, lineterminator='\n')
with zipfile.ZipFile('prediction_softblend_5050.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')
print("✅ Đã khôi phục: prediction_softblend_5050.zip")

# ==========================================================
# PHƯƠNG ÁN 2: 3-PARAM NELDER-MEAD
# ==========================================================
def kappa_loss_3param(coef, X, y):
    thresholds = np.array([coef[0]]*6 + [coef[1]]*7 + [coef[2]]*5)
    preds = (X > thresholds).sum(axis=1)
    return -cohen_kappa_score(y, preds, weights='quadratic')

result_3p = optimize.minimize(
    partial(kappa_loss_3param, X=blended_oof_logits, y=y_oof),
    x0=np.zeros(3), method='nelder-mead',
    options={'xatol': 1e-4, 'fatol': 1e-4, 'maxiter': 50000}
)
thresholds_3p = np.array([result_3p.x[0]]*6 + [result_3p.x[1]]*7 + [result_3p.x[2]]*5)

# Cộng 1 để đưa về thang điểm 1-19
final_preds_3p = np.clip((blended_test_logits > thresholds_3p).sum(axis=1) + 1, 1, 19)

pd.DataFrame({'Sentence ID': test_ids, 'Prediction': final_preds_3p}) \
  .to_csv('prediction', index=False, lineterminator='\n')
with zipfile.ZipFile('prediction_3param_nelder_mead.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')
print(f"✅ Đã khôi phục: prediction_3param_nelder_mead.zip (OOF QWK: {-result_3p.fun:.4f})")

# ==========================================================
# PHƯƠNG ÁN 3: RIDGE REGRESSION STACKING
# ==========================================================
meta_ridge = Ridge(alpha=10.0)
meta_ridge.fit(X_oof_scaled, y_oof)

# Cộng 1 để đưa từ nhãn 0-18 về thang 1-19
final_preds_ridge = np.clip(np.round(meta_ridge.predict(X_test_scaled)).astype(int) + 1, 1, 19)

pd.DataFrame({'Sentence ID': test_ids, 'Prediction': final_preds_ridge}) \
  .to_csv('prediction', index=False, lineterminator='\n')
with zipfile.ZipFile('prediction_ridge_stacking.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')
print("✅ Đã khôi phục: prediction_ridge_stacking.zip")

# ==========================================================
# PHƯƠNG ÁN 4: LIGHTGBM + SHALLOW FEATURES
# ==========================================================
arabic_diacritics = re.compile(r'[\u0651\u064e\u064b\u064f\u064c\u0650\u064d\u0652]')
def calculate_diacritics_ratio(text):
    if not isinstance(text, str) or len(text) == 0: return 0.0
    return len(re.findall(arabic_diacritics, text)) / len(text)

# Tính lại features bề mặt an toàn
df_all['Word_Count'] = df_all['Sentence'].apply(lambda x: len(str(x).split()))
df_blind_test['Word_Count'] = df_blind_test['Sentence'].apply(lambda x: len(str(x).split()))
df_all['Diacritics_Ratio'] = df_all['Sentence'].apply(calculate_diacritics_ratio)
df_blind_test['Diacritics_Ratio'] = df_blind_test['Sentence'].apply(calculate_diacritics_ratio)

X_lgb_train = np.column_stack([X_oof_scaled, df_all['Word_Count'].values, df_all['Diacritics_Ratio'].values])
X_lgb_test = np.column_stack([X_test_scaled, df_blind_test['Word_Count'].values, df_blind_test['Diacritics_Ratio'].values])

lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
lgb_model.fit(X_lgb_train, y_oof)

# Cộng 1 để đưa từ nhãn 0-18 về thang 1-19
final_preds_lgb = np.clip(np.round(lgb_model.predict(X_lgb_test)).astype(int) + 1, 1, 19)

pd.DataFrame({'Sentence ID': test_ids, 'Prediction': final_preds_lgb}) \
  .to_csv('prediction', index=False, lineterminator='\n')
with zipfile.ZipFile('prediction_lightgbm_features.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('prediction', arcname='prediction')
print("✅ Đã khôi phục: prediction_lightgbm_features.zip")

print("\n🎉 HOÀN TẤT! Cả 4 file zip đã sẵn sàng trong thư mục hiện tại để bạn nộp bài.")

🚀 ĐANG TÁI TẠO LẠI CÁC PHƯƠNG ÁN SUBMIT TỪ LOGITS...

✅ Đã khôi phục: prediction_softblend_5050.zip
✅ Đã khôi phục: prediction_3param_nelder_mead.zip (OOF QWK: 0.8468)
✅ Đã khôi phục: prediction_ridge_stacking.zip
✅ Đã khôi phục: prediction_lightgbm_features.zip

🎉 HOÀN TẤT! Cả 4 file zip đã sẵn sàng trong thư mục hiện tại để bạn nộp bài.
